In [64]:
import Gmsh: gmsh
using Gridap, GridapGmsh
using Random, LinearAlgebra, FFTW
using SpecialFunctions  # för gamma, besselk

In [65]:
u = zeros(5,5)

for a in 1:5, b in 1:5
    if a in 1:3
        u[a,b] = 2
    elseif b in 2:5
        u[a,b] = 4
    else
        u[a,b] = 5
    end
end
println(u)

fftu = fft(u)
println(fftu)
rng = MersenneTwister(1)
ξ = (randn(rng, 5, 5) .+ im*randn(rng, 5, 5)) ./ sqrt(2)
Y = real(ifft(sqrt.(fftu) .* ξ)) *5
println(ξ)
println(Y)

[2.0 2.0 2.0 2.0 2.0; 2.0 2.0 2.0 2.0 2.0; 2.0 2.0 2.0 2.0 2.0; 5.0 4.0 4.0 4.0 4.0; 5.0 4.0 4.0 4.0 4.0]
ComplexF64[72.0 + 0.0im 2.0 + 0.0im 2.0 + 0.0im 2.0 + 0.0im 2.0 + 0.0im; -5.5 + 16.927259454463893im -0.5 + 1.538841768587627im -0.5 + 1.538841768587627im -0.5 + 1.538841768587627im -0.5 + 1.538841768587627im; -5.5 - 3.995983904029484im -0.5 - 0.3632712640026803im -0.5 - 0.3632712640026803im -0.5 - 0.3632712640026803im -0.5 - 0.3632712640026803im; -5.5 + 3.995983904029484im -0.5 + 0.3632712640026803im -0.5 + 0.3632712640026803im -0.5 + 0.3632712640026803im -0.5 + 0.3632712640026803im; -5.5 - 16.927259454463893im -0.5 - 1.538841768587627im -0.5 - 1.538841768587627im -0.5 - 1.538841768587627im -0.5 - 1.538841768587627im]
ComplexF64[-0.09681382081338143 - 0.36878720827405603im 0.7410500117271404 + 0.07159954657603962im 0.2750924262978724 - 0.7465367456273365im -1.7331137680275708 + 0.8485995604804284im -1.4749986548846337 + 1.1901953955641666im; 0.6788660404316931 + 0.2846578492913397

In [66]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square")
    gmsh.model.geo.addPoint(0,0,0,1,1) # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1,0,0,1,2)
    gmsh.model.geo.addPoint(1,1,0,1,3)
    gmsh.model.geo.addPoint(0,1,0,1,4)
    gmsh.model.geo.addLine(1,2,1)
    gmsh.model.geo.addLine(2,3,2) # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3,4,3)
    gmsh.model.geo.addLine(4,1,4)
        
    gmsh.model.geo.addCurveLoop([1,2,3,4],1)
        
    gmsh.model.geo.addPlaneSurface([1],1)

    N = Int(round(1/h))  # number of elements per edge

    gmsh.model.geo.mesh.setTransfiniteCurve(1, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(2, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(3, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(4, N+1)

    #triangulation
    gmsh.model.geo.mesh.setTransfiniteSurface(1)

    gmsh.model.geo.synchronize()

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")
    gmsh.model.mesh.generate(2)

    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [67]:
p = 8
h = 1/(2^p)
N = Int(round(1/h)) #elements (lines between points)
Pₙ = N + 1 # antal punkter
order = 1

model = create_square_model(h)
reffe = ReferenceFE(lagrangian, Float64, order)
V = FESpace(model, reffe; conformity=:H1)

Ω  = Triangulation(model)
dΩ = Measure(Ω, 2*order)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00312409s, CPU 0.00441s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Transfinite)
Info    : Done meshing 2D (Wall 0.445511s, CPU 0.445463s)
Info    : 66049 nodes 132100 elements


GenericMeasure()

In [68]:
function matern_rho(d, σ, v, p)
    return σ^2 * (2.0^(1-v) / gamma(v)) * ((sqrt(2*v) * d / p)^v) * besselk(v, (sqrt(2*v) * d / p))
end

function delta(a, h, Pₙ)
    if a in 1:Pₙ
        return (a-1)*h
    elseif a in Pₙ+1:2*(Pₙ-1)
        return (2*(Pₙ-1)-(a-1))*h
    end
end


function sample_circulant(Pₙ, Δt, σ, v, p, h, rng)
    M = 2*(Pₙ-1)
    C = zeros(M,M)
    for a in 1:M, b in 1:M
        dx = delta(a, h, Pₙ)
        dy = delta(b, h, Pₙ)
        C[a,b] = Δt *matern_rho(√(dx^2+dy^2),σ,v,p)
    end

    Λ = real(fft(C))
    ξ = (randn(rng, M, M) .+ im*randn(rng, M, M)) ./ sqrt(2)
    Y = real(ifft(sqrt.(Λ) .* ξ)) * sqrt(M*M)
    return Y[1:Pₙ, 1:Pₙ]
end


sample_circulant (generic function with 2 methods)

In [69]:
Δt = 1e-5
σ  = 1.0
vp = 1.2 
p  = 0.15
rng = MersenneTwister(1)


vgrid = sample_circulant(Pₙ, Δt, σ, vp, p, h, rng)

vfun = interpolate_everywhere(x -> begin
    i = clamp(round(Int,x[1]*N)+1,1,Pₙ)
    j = clamp(round(Int,x[2]*N)+1,1,Pₙ)
    vgrid[i,j]
end, V)

v = get_free_dof_values(vfun)


#för pde solvern sen
a(u,v) = ∫(u*v)dΩ
Mmat = assemble_matrix(a, V, V)

w = Mmat * vgrid

#visualisering nu
uh_noise = FEFunction(V, v)
writevtk(Ω, "Ih_noise", cellfields=["IhΔW" => uh_noise])

LoadError: DimensionMismatch: second dimension of A, 66049, does not match the first dimension of B, 257